# Ñ contra 16 bibliotecas de otros lenguajes, en las dos T4 de Kaggle

Las mismas seis pruebas (`matmul` 1024 y 2048, `seno`, `x*2+1` y `suma` de 2^24 números, `softmax` 4096×1024 y `fft` de 2^20) con **los mismos datos** en Ñ (C1EÑ en jit, nativo y GPU) y en:

| lenguaje | biblioteca | aquí | por defecto |
|---|---|---|---|
| Python | PyTorch | CPU y GPU | **activo** |
| C++ | Eigen | CPU | **activo** |
| C | OpenBLAS | CPU (BLAS solo hace matmul; el resto, bucles C con OpenMP) | **activo** |
| JavaScript | TensorFlow.js (tfjs-node) | CPU, **float32** | **activo** |
| Go | Gorgonia | CPU | **activo** |
| Fortran | Intel MKL | CPU | **activo** |
| Julia | CUDA.jl | CPU y GPU | apagado (instalar ~10–15 min) |
| Rust | Candle | CPU y GPU | apagado (compilar con CUDA ~15–25 min) |
| R | torch | CPU y GPU | apagado (~2 GB) |
| Java | ND4J | CPU | apagado |
| MATLAB | **Octave en su lugar** (MATLAB necesita licencia) | CPU | apagado |
| C# | TorchSharp | CPU y GPU | apagado (~2,5 GB) |
| Swift | **Swift sin Metal** (Metal solo existe en Apple) | CPU, un hilo | apagado |
| Scala | Breeze | CPU | apagado |
| Kotlin | KotlinDL (se mide TensorFlow 1.15 para Java, su motor) | CPU | apagado |

### Cómo se reparte el trabajo (las dos T4 a la vez)

1. **Instalar y compilar, varios a la vez.** Es lo que más tarda (bajar Julia, compilar Rust, libtorch de R y de .NET…) y casi todo es esperar a la red o al compilador, así que se hacen `INSTALAR_A_LA_VEZ` (3) lenguajes en paralelo. `apt-get` va de uno en uno porque no admite dos a la vez.
2. **Medir, dos a la vez, uno por GPU.** Dos trabajadores sacan lenguajes de una cola: el 1 tiene la **T4 0** y la mitad de los núcleos de la CPU, el 2 la **T4 1** y la otra mitad. Cada programa solo ve su GPU (`CUDA_VISIBLE_DEVICES`) y queda atado a sus núcleos (`taskset`), así que las dos medidas no se estorban. Cuando uno termina, toma el siguiente de la cola, empezando por los más largos.

**Lo que cuesta, dicho claro:** en `MODO = 'paralelo'` cada lenguaje tiene la mitad de la CPU (un núcleo físico de los dos de Kaggle), así que sus tiempos de **CPU** son los de esa mitad, para todos por igual; los de **GPU** son los de una T4 entera. La tabla dice el modo y los hilos. Con `MODO = 'secuencial'` va uno detrás de otro con toda la CPU y la T4 0: tarda más, y los tiempos de CPU son los de la máquina completa.

Cada uno escribe `RES,prueba,tamaño,dispositivo,ms,huella,precisión`. La **huella** (suma del resultado) se compara con la de NumPy en doble precisión. Si un lenguaje falla al instalarse o al correr, se salta, se apunta por qué y el resto sigue.

**Por qué algunos solo en CPU:** Eigen, OpenBLAS, MKL, Gorgonia y Breeze son de CPU. TensorFlow.js para GPU, ND4J para CUDA y TensorFlow 1.15 (KotlinDL) piden CUDA 10/11, y Kaggle tiene CUDA 12. MATLAB y Metal no se pueden instalar aquí: sus sustitutos están marcados como tales.

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2* e *Internet → On*.

In [ ]:
# 1 · Qué corre y cómo. Cambia y vuelve a correr desde aquí.
ESCALA = 'normal'            # 'chica' para probar que todo corre en pocos minutos
MODO = 'paralelo'            # 'paralelo': dos a la vez, uno por GPU · 'secuencial': uno detrás de otro, CPU entera
INSTALAR_A_LA_VEZ = 3        # cuántos lenguajes se instalan/compilan a la vez
ACTIVOS = {
    'Ñ': True,
    'PyTorch': True, 'Eigen': True, 'OpenBLAS': True, 'TensorFlow.js': True, 'Gorgonia': True, 'MKL': True,
    'CUDA.jl': False, 'Candle': False, 'torch (R)': False, 'ND4J': False, 'Octave (en lugar de MATLAB)': False,
    'TorchSharp': False, 'Swift (sin Metal)': False, 'Breeze': False, 'KotlinDL': False,
}
TRABAJO = '/kaggle/working'

In [ ]:
# 2 · Node 22 y el proyecto (dataset subido, o git clone)
import os, subprocess, shutil, re, glob, time, json
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {TRABAJO}', shell=True, check=True)
    os.environ['PATH'] = f'{TRABAJO}/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())
PROYECTO = f'{TRABAJO}/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO): print('ya estaba en', PROYECTO)
elif candidatos:
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules')); print('copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
os.chdir(PROYECTO)
COMP = f'{PROYECTO}/c1en/comparar'
if not os.path.exists(f'{COMP}/n.esl'):
    commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
    raise SystemExit(f'falta c1en/comparar/ (los programas de la comparación). El repositorio está en «{commit}»: haz git push desde tu PC.')
print(subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!lscpu | grep -E '^(Model name|CPU\(s\)|Thread|Core)'

In [ ]:
# 3 · Ayudantes: instalar (con candado para apt), correr un programa y quedarse con sus líneas RES
import threading, pandas as pd
from concurrent.futures import ThreadPoolExecutor
ARG = ' '.join(['chica'] if ESCALA == 'chica' else [])
filas, estado, tiempos = [], {}, {}
candado_filas, candado_apt, candado_print = threading.Lock(), threading.Lock(), threading.Lock()
def decir(*a):
    with candado_print: print(*a, flush=True)
def sh(cmd, timeout=3600, env=None, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd,
                       env=dict(os.environ, **(env or {})))
    if r.returncode:
        raise RuntimeError(f'«{cmd[:140]}» salió con {r.returncode}:\n' + '\n'.join((r.stdout + r.stderr).strip().splitlines()[-15:]))
    return r.stdout
def apt(*paquetes):
    with candado_apt:                                       # apt-get no admite dos a la vez
        sh('apt-get update -qq', timeout=900)
        sh('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends ' + ' '.join(paquetes), timeout=1800)
HAY_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
N_GPU = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()) if HAY_GPU else 0

# Los núcleos físicos (los hilos hermanos juntos), para repartirlos entre los dos trabajadores
nucleos = {}
for c in sorted(glob.glob('/sys/devices/system/cpu/cpu[0-9]*')):
    try:
        cid = (open(f'{c}/topology/physical_package_id').read().strip(), open(f'{c}/topology/core_id').read().strip())
        nucleos.setdefault(cid, []).append(int(c.rsplit('cpu', 1)[1]))
    except OSError: pass
grupos = list(nucleos.values()) or [[i] for i in range(os.cpu_count())]
if MODO == 'paralelo' and N_GPU >= 2 and len(grupos) >= 2:
    mitad = len(grupos) // 2
    TRABAJADORES = [dict(id=1, gpu=0, cpus=sorted(sum(grupos[:mitad], []))), dict(id=2, gpu=1, cpus=sorted(sum(grupos[mitad:], [])))]
else:
    if MODO == 'paralelo': print('(sin dos GPU o sin núcleos que repartir: va en secuencial)')
    TRABAJADORES = [dict(id=1, gpu=0, cpus=sorted(sum(grupos, [])))]
for t in TRABAJADORES:
    h = len(t['cpus'])
    t['env'] = {'CUDA_VISIBLE_DEVICES': str(t['gpu']), 'C1EN_GPU': '0', 'HILOS': str(h), 'OMP_NUM_THREADS': str(h),
                'MKL_NUM_THREADS': str(h), 'OPENBLAS_NUM_THREADS': str(h), 'GOMAXPROCS': str(h), 'RAYON_NUM_THREADS': str(h),
                'JULIA_NUM_THREADS': str(h)} if len(TRABAJADORES) > 1 else {'CUDA_VISIBLE_DEVICES': str(t['gpu']), 'C1EN_GPU': '0'}
    t['prefijo'] = f"taskset -c {','.join(map(str, t['cpus']))} " if len(TRABAJADORES) > 1 and shutil.which('taskset') else ''
    print(f"trabajador {t['id']}: GPU {t['gpu']} · CPU {t['cpus']}")
print('GPU:', N_GPU, '· escala', ESCALA, '· modo', MODO if len(TRABAJADORES) > 1 else 'secuencial')

## Los lenguajes: cómo se instala cada uno y qué se mide

Cada lenguaje tiene su `instalar()` (que deja lo que hace falta y devuelve rutas y variables) y sus tareas de medida (la orden que se corre, en qué carpeta y si usa la GPU). Nada se corre en esta celda: solo se describe.

In [ ]:
L = {}
def lenguaje(nombre, lengua, biblioteca, estimado_s):
    def reg(f):
        L[nombre] = dict(nombre=nombre, lenguaje=lengua, biblioteca=biblioteca, estimado=estimado_s, fn=f)
        return f
    return reg
# instalar(): devuelve una lista de tareas (etiqueta, orden, env, cwd, usa_gpu)

@lenguaje('Ñ', 'Ñ', 'C1EÑ', 90)
def _():
    sh('npm install koffi --no-audit --no-fund --loglevel=error', timeout=900, cwd=PROYECTO)
    sh('node motor-exp/util/construir.js', cwd=PROYECTO)
    return [(f'Ñ · {d}', f'node motor-exp/util/n-node.js c1en/comparar/n.esl {d} {ARG}', {}, PROYECTO, d == 'gpu')
            for d in ['jit', 'nativo'] + (['gpu'] if HAY_GPU else [])]

@lenguaje('PyTorch', 'Python', 'PyTorch', 60)
def _():
    sh('python -c "import torch"')
    return [('Python · PyTorch', f'python {COMP}/pytorch.py {ARG}', {}, PROYECTO, True)]

@lenguaje('Eigen', 'C++', 'Eigen', 20)
def _():
    apt('libeigen3-dev', 'g++')
    sh(f'g++ -O3 -march=native -fopenmp -I/usr/include/eigen3 {COMP}/eigen.cpp -o {TRABAJO}/eigen')
    return [('C++ · Eigen', f'{TRABAJO}/eigen {ARG}', {}, TRABAJO, False)]

@lenguaje('OpenBLAS', 'C', 'OpenBLAS', 15)
def _():
    apt('libopenblas-dev', 'gcc')
    sh(f'gcc -O3 -march=native -fopenmp {COMP}/openblas.c -o {TRABAJO}/openblas -lopenblas -lm')
    return [('C · OpenBLAS', f'{TRABAJO}/openblas {ARG}', {}, TRABAJO, False)]

@lenguaje('MKL', 'Fortran', 'Intel MKL', 15)
def _():
    apt('gfortran')
    M = f'{TRABAJO}/mklenv'
    if not os.path.exists(f'{M}/bin/pip'): sh(f'python -m venv {M}')
    sh(f'{M}/bin/pip install -q mkl mkl-devel', timeout=1800)
    rt = sorted(glob.glob(f'{M}/lib/libmkl_rt.so.*'))
    if not rt: raise RuntimeError('no apareció libmkl_rt.so.* en el entorno de MKL')
    D = f'{TRABAJO}/mkl'; os.makedirs(D, exist_ok=True)
    sh(f'gfortran -O3 -march=native -fopenmp -I{M}/include {M}/include/mkl_dfti.f90 {COMP}/mkl.f90 '
       f'-L{M}/lib -l:{os.path.basename(rt[-1])} -Wl,-rpath,{M}/lib -o {D}/mklf', cwd=D)
    return [('Fortran · MKL', f'{D}/mklf {ARG}', {}, D, False)]

@lenguaje('TensorFlow.js', 'JavaScript', 'TensorFlow.js', 45)
def _():
    D = f'{TRABAJO}/tfjs'; os.makedirs(D, exist_ok=True)
    sh('npm init -y >/dev/null && npm install @tensorflow/tfjs-node@4.22.0 --no-audit --no-fund --loglevel=error', cwd=D, timeout=1800)
    shutil.copy(f'{COMP}/tfjs.js', D)
    return [('JavaScript · TensorFlow.js', f'node tfjs.js {ARG}', {}, D, False)]

@lenguaje('Gorgonia', 'Go', 'Gorgonia', 60)
def _():
    if not os.path.exists(f'{TRABAJO}/go/bin/go'):
        ver = json.loads(sh('curl -fsSL "https://go.dev/dl/?mode=json"'))[0]['version']
        sh(f'curl -fsSL https://go.dev/dl/{ver}.linux-amd64.tar.gz | tar -xz -C {TRABAJO}', timeout=900)
    env = {'PATH': f'{TRABAJO}/go/bin:' + os.environ['PATH'], 'GOPATH': f'{TRABAJO}/gopath', 'GOCACHE': f'{TRABAJO}/gocache', 'GOFLAGS': '-mod=mod'}
    D = f'{TRABAJO}/gorgonia'; os.makedirs(D, exist_ok=True)
    shutil.copy(f'{COMP}/gorgonia.go', f'{D}/main.go')
    if not os.path.exists(f'{D}/go.mod'): sh('go mod init comparar', env=env, cwd=D)
    sh('go get gorgonia.org/tensor@v0.9.24 && go build -o gorgonia .', env=env, cwd=D, timeout=1800)
    return [('Go · Gorgonia', f'./gorgonia {ARG}', {}, D, False)]

@lenguaje('CUDA.jl', 'Julia', 'CUDA.jl', 120)
def _():
    J = f'{TRABAJO}/julia-1.13.2/bin/julia'
    if not os.path.exists(J):
        sh(f'curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz | tar -xz -C {TRABAJO}', timeout=1800)
    env = {'JULIA_DEPOT_PATH': f'{TRABAJO}/julia-depot'}
    sh(f'{J} -e \'using Pkg; Pkg.add(["CUDA", "FFTW"]); using CUDA, FFTW\'', env=env, timeout=3600)
    if not HAY_GPU: env['SIN_GPU'] = '1'
    return [('Julia · CUDA.jl', f'{J} --threads=auto {COMP}/cudajl.jl {ARG}', env, TRABAJO, True)]

@lenguaje('Candle', 'Rust', 'Candle', 60)
def _():
    cargo = os.path.expanduser('~/.cargo/bin/cargo')
    if not os.path.exists(cargo): sh('curl -fsSL https://sh.rustup.rs | sh -s -- -y --profile minimal', timeout=1800)
    nvcc = shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if os.path.exists('/usr/local/cuda/bin/nvcc') else None)
    env = {'PATH': os.path.dirname(cargo) + ':/usr/local/cuda/bin:' + os.environ['PATH'], 'CUDA_COMPUTE_CAP': '75',
           'CARGO_TARGET_DIR': f'{TRABAJO}/candle-target'}
    D = f'{TRABAJO}/candle'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/candle', D)
    rasgos = '--features cuda' if (HAY_GPU and nvcc) else ''
    if HAY_GPU and not nvcc: decir('   Candle: sin nvcc, solo CPU')
    sh(f'cargo build --release {rasgos}', env=env, cwd=D, timeout=3600)
    return [('Rust · Candle', f'{TRABAJO}/candle-target/release/comparar-candle {ARG}', {}, D, bool(rasgos))]

@lenguaje('torch (R)', 'R', 'torch', 90)
def _():
    if not shutil.which('Rscript'):
        apt('software-properties-common', 'dirmngr', 'wget', 'gnupg')
        with candado_apt:
            sh('wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc > /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc')
            sh('add-apt-repository -y "deb https://cloud.r-project.org/bin/linux/ubuntu $(lsb_release -cs)-cran40/"', timeout=900)
        apt('r-base-core', 'r-base-dev')
    tipo = 'cu128' if HAY_GPU else 'cpu'
    instalar = ('options(timeout = 1800); '
                'ver <- available.packages(repos = "https://cloud.r-project.org")["torch", "Version"]; '
                f'options(repos = c(torch = sprintf("https://torch-cdn.mlverse.org/packages/{tipo}/%s/", ver), CRAN = "https://cloud.r-project.org")); '
                'install.packages("torch"); library(torch); if (!torch_is_installed()) install_torch()')
    sh(f"Rscript -e '{instalar}'", env={'TORCH_INSTALL': '1'}, timeout=3600)
    return [('R · torch', f'Rscript {COMP}/rtorch.R {ARG}', {}, TRABAJO, True)]

@lenguaje('TorchSharp', 'C#', 'TorchSharp', 90)
def _():
    DN = f'{TRABAJO}/dotnet'
    if not os.path.exists(f'{DN}/dotnet'):
        sh(f'curl -fsSL https://dot.net/v1/dotnet-install.sh -o {TRABAJO}/dotnet-install.sh && bash {TRABAJO}/dotnet-install.sh --channel 8.0 --install-dir {DN}', timeout=1800)
    env = {'DOTNET_ROOT': DN, 'PATH': DN + ':' + os.environ['PATH'], 'DOTNET_CLI_TELEMETRY_OPTOUT': '1', 'DOTNET_NOLOGO': '1',
           'NUGET_PACKAGES': f'{TRABAJO}/nuget'}
    D = f'{TRABAJO}/torchsharp'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/torchsharp', D)
    sh(f"dotnet add package {'TorchSharp-cuda-linux' if HAY_GPU else 'TorchSharp-cpu'}", env=env, cwd=D, timeout=3600)
    sh('dotnet build -c Release -o salida', env=env, cwd=D, timeout=1800)
    return [('C# · TorchSharp', f'dotnet salida/comparar.dll {ARG}', env, D, True)]

def jvm():
    if not shutil.which('javac'): apt('openjdk-17-jdk-headless')
    cs = f'{TRABAJO}/cs'
    with candado_apt:                                        # que dos JVM no bajen Coursier a la vez
        if not os.path.exists(cs):
            sh(f'curl -fsSL https://github.com/coursier/launchers/raw/master/cs-x86_64-pc-linux.gz | gzip -d > {cs}.tmp && chmod +x {cs}.tmp && mv {cs}.tmp {cs}', timeout=900)
    return cs
def jhilos(): return '-XX:ActiveProcessorCount=$HILOS ' if len(TRABAJADORES) > 1 else ''

@lenguaje('ND4J', 'Java', 'ND4J', 60)
def _():
    cs = jvm()
    cp = sh(f'{cs} fetch --classpath org.nd4j:nd4j-native:1.0.0-M2.1 org.nd4j:nd4j-native:1.0.0-M2.1,classifier=linux-x86_64', timeout=1800).strip()
    return [('Java · ND4J', f'java -Xmx12g {jhilos()}-cp "{cp}" {COMP}/Nd4jBench.java {ARG}', {}, TRABAJO, False)]

@lenguaje('Breeze', 'Scala', 'Breeze', 60)
def _():
    jvm(); apt('libopenblas-dev')                  # la libblas.so.3 que encuentra netlib
    sc = f'{TRABAJO}/scala-cli'
    if not os.path.exists(sc):
        sh(f'curl -fsSL https://github.com/VirtusLab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gzip -d > {sc} && chmod +x {sc}', timeout=900)
    sh(f'{sc} compile --server=false {COMP}/breeze.sc', timeout=1800)           # baja Scala y Breeze aquí, no al medir
    return [('Scala · Breeze', f'{sc} run --server=false {COMP}/breeze.sc -- {ARG}', {}, TRABAJO, False)]

@lenguaje('KotlinDL', 'Kotlin', 'KotlinDL', 60)
def _():
    cs = jvm()
    K = f'{TRABAJO}/kotlinc/bin/kotlinc'
    if not os.path.exists(K):
        rel = json.loads(sh('curl -fsSL https://api.github.com/repos/JetBrains/kotlin/releases/latest'))
        url = next(a['browser_download_url'] for a in rel['assets'] if re.match(r'kotlin-compiler-[\d.]+\.zip$', a['name']))
        sh(f'curl -fsSL -o {TRABAJO}/kotlinc.zip {url} && cd {TRABAJO} && python -m zipfile -e kotlinc.zip . && chmod +x kotlinc/bin/*', timeout=1800)
    cp = sh(f'{cs} fetch --classpath org.jetbrains.kotlinx:kotlin-deeplearning-tensorflow:0.5.2', timeout=1800).strip()
    sh(f'{K} {COMP}/kotlindl.kt -cp "{cp}" -include-runtime -d {TRABAJO}/kotlindl.jar', timeout=1800)
    return [('Kotlin · KotlinDL', f'java -Xmx12g {jhilos()}-cp "{TRABAJO}/kotlindl.jar:{cp}" KotlindlKt {ARG}', {}, TRABAJO, False)]

@lenguaje('Octave (en lugar de MATLAB)', 'MATLAB (Octave)', 'Octave', 40)
def _():
    apt('octave', 'libopenblas0')
    return [('MATLAB → Octave', f'octave-cli --no-gui {COMP}/octave.m {ARG}', {}, TRABAJO, False)]

@lenguaje('Swift (sin Metal)', 'Swift', 'Swift puro, sin Metal', 150)
def _():
    S = os.path.expanduser('~/.local/share/swiftly')
    if not shutil.which('swiftc') and not os.path.exists(f'{S}/bin/swiftc'):
        apt('binutils', 'git', 'gnupg2', 'libc6-dev', 'libcurl4-openssl-dev', 'libedit2', 'libpython3-dev', 'libsqlite3-0',
            'libxml2-dev', 'libz3-dev', 'pkg-config', 'tzdata', 'unzip', 'zlib1g-dev', 'libstdc++-12-dev', 'libgcc-12-dev')
        sh(f'cd {TRABAJO} && curl -fsSLO https://download.swift.org/swiftly/linux/swiftly-$(uname -m).tar.gz && tar zxf swiftly-$(uname -m).tar.gz '
           f'&& ./swiftly init --assume-yes --quiet-shell-followup', timeout=3600)
    env = {'PATH': f'{S}/bin:' + os.environ['PATH']}
    sh(f'swiftc -Ounchecked {COMP}/swift.swift -o {TRABAJO}/swiftbench', env=env, timeout=1800)
    return [('Swift (sin Metal)', f'{TRABAJO}/swiftbench {ARG}', {}, TRABAJO, False)]

print(len(L), 'lenguajes descritos ·', sum(1 for k in L if ACTIVOS.get(k)), 'activos:', ', '.join(k for k in L if ACTIVOS.get(k)))

## Fase 1 · Instalar y compilar, varios a la vez

Lo que más tarda es bajar y compilar. Se hace en paralelo (`INSTALAR_A_LA_VEZ`). Cada lenguaje dice al terminar si quedó listo o por qué no.

In [ ]:
TAREAS = []          # (estimado, etiqueta, lenguaje, biblioteca, orden, env, cwd, usa_gpu)
def instalar(nombre):
    d = L[nombre]; t0 = time.time()
    try:
        tareas = d['fn']()
        tiempos[nombre] = {'instalar_s': round(time.time() - t0)}
        with candado_filas:
            for (et, cmd, env, cwd, gpu) in tareas:
                if gpu and not HAY_GPU: gpu = False
                TAREAS.append((d['estimado'] / len(tareas), et, d['lenguaje'], d['biblioteca'], cmd, env, cwd, gpu, nombre))
        decir(f'✓ {nombre}: listo en {time.time() - t0:.0f} s')
    except Exception as e:
        estado[nombre] = '✗ al instalar: ' + str(e).splitlines()[0][:200]
        decir(f'✗ {nombre}: no se pudo instalar ({time.time() - t0:.0f} s)\n{e}')
for k in L:
    if not ACTIVOS.get(k): estado[k] = 'apagado'
t0 = time.time()
with ThreadPoolExecutor(max_workers=INSTALAR_A_LA_VEZ) as ex:
    list(ex.map(instalar, [k for k in L if ACTIVOS.get(k)]))
print(f'\nfase 1: {time.time() - t0:.0f} s de pared · {len(TAREAS)} tareas de medida listas')

## Fase 2 · Medir, uno por GPU

Dos trabajadores (uno por T4, cada uno con su mitad de la CPU) sacan tareas de la cola, de la más larga a la más corta. La salida de cada tarea se escribe entera cuando termina, con el trabajador delante.

In [ ]:
import queue
cola = queue.Queue()
for t in sorted(TAREAS, key=lambda t: -t[0]): cola.put(t)
def correr_tarea(trab, tarea):
    _, et, lengua, bib, cmd, env, cwd, gpu, nombre = tarea
    t0 = time.time()
    salida = sh(trab['prefijo'] + cmd, timeout=5400, env=dict(env, **trab['env']), cwd=cwd)
    nuevas = []
    for l in salida.splitlines():
        if not l.startswith('RES,'): continue
        _, prueba, tam, disp, ms, huella, prec = l.strip().split(',')
        nuevas.append(dict(entrada=et, lenguaje=lengua, biblioteca=bib, prueba=prueba, tam=tam, dispositivo=disp, ms=float(ms),
                           huella=float(huella), precision=prec, trabajador=trab['id'], gpu_fisica=trab['gpu'] if disp == 'gpu' else None,
                           hilos=len(trab['cpus']), modo='paralelo' if len(TRABAJADORES) > 1 else 'secuencial'))
    with candado_filas:
        filas.extend(nuevas)
        tiempos.setdefault(nombre, {})[et] = round(time.time() - t0)
    comentarios = [l for l in salida.splitlines() if l.startswith('#')]
    decir(f"[T{trab['id']}·GPU{trab['gpu']}] ✓ {et}: {len(nuevas)} medidas en {time.time() - t0:.0f} s" + ''.join('\n      ' + c for c in comentarios))
    return len(nuevas)
def trabajador(trab):
    while True:
        try: tarea = cola.get_nowait()
        except queue.Empty: return
        nombre = tarea[8]
        try:
            n = correr_tarea(trab, tarea)
            with candado_filas:
                estado[nombre] = estado.get(nombre, '') if estado.get(nombre, '').startswith('✗') else '✓'
        except Exception as e:
            with candado_filas: estado[nombre] = f'✗ al medir {tarea[1]}: ' + str(e).splitlines()[0][:200]
            decir(f"[T{trab['id']}·GPU{trab['gpu']}] ✗ {tarea[1]}:\n{e}")
t0 = time.time()
hilos_ = [threading.Thread(target=trabajador, args=(t,)) for t in TRABAJADORES]
for h in hilos_: h.start()
for h in hilos_: h.join()
print(f'\nfase 2: {time.time() - t0:.0f} s de pared con {len(TRABAJADORES)} trabajador(es) · {len(filas)} medidas')
suma_tareas = sum(v for d in tiempos.values() for k, v in d.items() if k != 'instalar_s')
print(f'(una detrás de otra habrían sido ~{suma_tareas} s)')

## ¿Calculan lo mismo? La huella contra NumPy

Cada huella se compara con la de NumPy en doble precisión (`c1en/comparar/referencia.py`). En doble precisión la diferencia relativa tiene que ser pequeñísima (≤ 1e-9; las sumas de 16 millones de números, en distinto orden, se quedan en ~1e-12); en float32 (TensorFlow.js, la FFT de KotlinDL), ~1e-4.

In [ ]:
import importlib.util, numpy as np
spec = importlib.util.spec_from_file_location('referencia', f'{COMP}/referencia.py'); ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
REF = ref.huellas((128, 256), 65536, 256, 1024, 16384) if ESCALA == 'chica' else ref.huellas()
df = pd.DataFrame(filas)
df['referencia'] = [REF.get((p, t), np.nan) for p, t in zip(df['prueba'], df['tam'])]
df['dif_rel'] = (df['huella'] - df['referencia']).abs() / df['referencia'].abs()
df['bien'] = np.where(df['precision'] == 'f32', df['dif_rel'] < 1e-3, df['dif_rel'] < 1e-9)
malos = df[~df['bien']]
print(f'{len(df)} medidas · {int(df["bien"].sum())} con la huella de NumPy · {len(malos)} distintas')
if len(malos): print(malos[['entrada', 'prueba', 'tam', 'dispositivo', 'huella', 'referencia', 'dif_rel']].to_string(index=False))
print('\nEstado y tiempos de cada lenguaje:')
for k in L:
    print(f'  {k:30s} {estado.get(k, "?"):12s} {tiempos.get(k, "")}')

## La tabla

Milisegundos, la mejor de 3 (menos es mejor). Cada columna es *lenguaje · biblioteca (dispositivo)*. En modo paralelo los tiempos de CPU son con la mitad de la CPU (lo dice la cabecera); los de GPU, con una T4 entera. Debajo, para cada prueba, quién va primero, cuántas veces más lento va cada uno y el puesto de Ñ.

In [ ]:
modo = df['modo'].iloc[0] if len(df) else MODO
hilos_cpu = sorted(int(h) for h in df['hilos'].unique()) if len(df) else []
print(f'modo {modo} · hilos de CPU por lenguaje: {hilos_cpu}\n')
df['col'] = df['entrada'].str.replace('Ñ · ', 'Ñ ', regex=False) + df.apply(lambda r: '' if r['lenguaje'] == 'Ñ' else f" ({r['dispositivo']})", axis=1)
orden_tam = lambda s: s.map(lambda t: float(str(t).split('x')[0]))
ancho = df.pivot_table(index=['prueba', 'tam'], columns='col', values='ms', aggfunc='min')
ancho = ancho.reset_index().sort_values(['prueba', 'tam'], key=lambda s: orden_tam(s) if s.name == 'tam' else s).set_index(['prueba', 'tam'])
pd.set_option('display.width', 400); pd.set_option('display.max_columns', 60)
print(ancho.round(2).T.to_string())
print()
for (p, t), g in df.groupby(['prueba', 'tam'], sort=False):
    g = g.sort_values('ms'); mejor = g['ms'].iloc[0]
    linea = ' · '.join(f"{r['col']} {r['ms']:.1f}" + ('' if k == 0 else f" (×{r['ms'] / mejor:.1f})") for k, (_, r) in enumerate(g.iterrows()))
    pos = [k + 1 for k, (_, r) in enumerate(g.iterrows()) if r['lenguaje'] == 'Ñ']
    print(f'{p} {t}: Ñ en el puesto {pos} de {len(g)}\n   {linea}\n')

In [ ]:
import matplotlib.pyplot as plt
pruebas = list(dict.fromkeys(zip(df['prueba'], df['tam'])))
fig, ejes = plt.subplots(len(pruebas), 1, figsize=(14, 3.2 * len(pruebas)))
ejes = np.atleast_1d(ejes)
for ax, (p, t) in zip(ejes, pruebas):
    g = df[(df['prueba'] == p) & (df['tam'] == t)].sort_values('ms')
    colores = ['#c0392b' if l == 'Ñ' else ('#2e86c1' if d == 'gpu' else '#7f8c8d') for l, d in zip(g['lenguaje'], g['dispositivo'])]
    ax.barh(g['col'], g['ms'], color=colores); ax.set_xscale('log'); ax.invert_yaxis()
    ax.set_title(f'{p} · {t}  (ms, escala log; rojo Ñ, azul GPU, gris CPU; modo {modo})', fontsize=10); ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()

mm = df[df['prueba'] == 'matmul'].copy()
mm['GFLOP/s'] = 2 * mm['tam'].astype(float) ** 3 / (mm['ms'] / 1000) / 1e9
print(mm.pivot_table(index='col', columns='tam', values='GFLOP/s').sort_values(mm['tam'].max(), ascending=False).round(1).to_string())

In [ ]:
# Los CSV para descargar (Output) — y para mandármelos
df.drop(columns=['col']).to_csv(f'{TRABAJO}/comparar-lenguajes.csv', index=False)
ancho.to_csv(f'{TRABAJO}/comparar-lenguajes-resumen.csv')
json.dump(dict(escala=ESCALA, modo=modo, trabajadores=[{k: t[k] for k in ('id', 'gpu', 'cpus')} for t in TRABAJADORES], estado=estado, tiempos=tiempos,
               commit=subprocess.run(['git', 'log', '-1', '--format=%h'], capture_output=True, text=True).stdout.strip()),
          open(f'{TRABAJO}/comparar-lenguajes-info.json', 'w'), ensure_ascii=False, indent=1)
for f in ['comparar-lenguajes.csv', 'comparar-lenguajes-resumen.csv', 'comparar-lenguajes-info.json']:
    print(f'{TRABAJO}/{f}: {os.path.getsize(f"{TRABAJO}/{f}") / 1024:.1f} KB')

## Cómo leerlo

- **Lo justo de la comparación:** todos en doble precisión salvo TensorFlow.js (float32) y la FFT de KotlinDL; todos con los datos ya en su dispositivo y el resultado sin copiar. PyTorch, R torch y TorchSharp usan cuBLAS y cuFFT en la GPU (con FMA); C1EÑ no usa FMA para dar los mismos bits que la CPU, así que en `matmul` de GPU su techo es la mitad.
- **Modo paralelo:** cada lenguaje tuvo su propia T4 y la mitad de la CPU, atado a sus núcleos: no se estorban, y todos compiten en las mismas condiciones. Para tiempos de CPU con la máquina entera, `MODO = 'secuencial'`.
- **CPU:** Eigen, OpenBLAS, MKL y PyTorch usan AVX/FMA y todos sus núcleos; los de la JVM y Go dependen de su BLAS; Swift va en un hilo a propósito (es Swift puro).
- **Lo que no está:** OpenBLAS, Gorgonia, Candle, ND4J y Swift no traen FFT.
- Si un lenguaje sale ✗ en el estado, su error está en la fase 1 o 2: mándamelo y lo arreglo (C++, C, Fortran, Octave y Ñ se probaron antes de entregarlos, TensorFlow.js con su motor de JavaScript puro; los demás se prueban aquí por primera vez).